# Реальная лабораторная 03 — Conversion и Bank Marketing

Сравнение двух бинарных целей и аудит момента доступности признаков.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## Часть A. Online Shoppers Purchasing Intention

In [ ]:
shop = load_real("online_shoppers")
print(shop.shape); display(shop.head()); display(shop.dtypes.to_frame("dtype"))
target = next((c for c in shop.columns if c.lower()=="revenue"), None)
assert target, "Не найден target Revenue"
shop[target] = shop[target].astype(str).str.lower().isin(["true","1","yes"])
print(shop[target].value_counts(normalize=True).rename("share"))

### Конверсия по сегментам

In [ ]:
for col in ["Month","VisitorType","Weekend","SpecialDay","TrafficType"]:
    if col in shop.columns:
        display(shop.groupby(col, dropna=False)[target].agg(["mean","count"]).sort_values("mean", ascending=False).head(20))
num = shop.select_dtypes(include=np.number).columns.tolist()
if num:
    display(shop.groupby(target)[num].median().T.rename(columns={False:"no_purchase_median",True:"purchase_median"}))

### Диагностика точки прогнозирования

In [ ]:
# PageValues агрегирует ценность просмотренных страниц; доступность зависит от момента принятия решения.
for col in ["PageValues","ExitRates","BounceRates","ProductRelated_Duration"]:
    if col in shop.columns:
        print(col, shop[col].describe(percentiles=[.5,.9,.95,.99]).to_dict())
print("Перед использованием каждого признака ответь: известен ли он до момента, когда требуется прогноз?")

## Часть B. Bank Marketing

In [ ]:
bank = load_real("bank_marketing")
print(bank.shape); display(bank.head()); display(bank.dtypes.to_frame("dtype"))
target_b = next((c for c in bank.columns if c.lower()=="y"), None)
assert target_b, "Не найден target y"
bank[target_b] = bank[target_b].astype(str).str.lower().str.strip()
print(bank[target_b].value_counts(normalize=True))

### Длительность звонка, повторные контакты и временная логика

In [ ]:
if "duration" in bank.columns:
    print("duration by outcome")
    display(bank.groupby(target_b)["duration"].describe())
    print("DURATION — постконтактная переменная: исключить, если прогноз делается до звонка.")
if "pdays" in bank.columns:
    print("pdays sentinel 999 share:", (pd.to_numeric(bank["pdays"], errors="coerce")==999).mean())
    bank["previous_contact_flag"] = (pd.to_numeric(bank["pdays"], errors="coerce") != 999)
if "month" in bank.columns:
    display(bank.groupby("month")[target_b].apply(lambda s:(s=="yes").mean()).to_frame("subscription_rate"))

## Самостоятельные задания

1. Сравни конверсию в online shoppers по типу посетителя и месяцу с доверительными интервалами.
2. Проведи аудит, какие признаки доступны в начале сессии, в середине и после завершения.
3. Для Bank Marketing сравни описательные выводы с/без duration. Не называй post-call корреляцию пригодным прогнозом до звонка.
4. Обсуди временной порядок Bank Marketing и возможность случайного split.
5. Сформулируй ограничения внешней применимости обоих наборов.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.